# Notebook 01_2 — Create Dataset (Text-Only (RoBERTa) Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Women's Shoes (Size 8)

---

Pipeline:
1. Load cleaned panel data (train + val splits)
2. Load pre-trained embeddings from Part 5 prediction zips
3. Compute PCA features (5 components)
4. Compute cluster similarity features (5 KMeans clusters)
5. Compute neighbor distances (5 nearest neighbors per product)
6. Compute weighted substitute prices (BLP-style IV for DoubleML)
7. Join all prediction outputs (level + diff, time-independent + lag1)
8. Save final train and val CSVs as zip files

**Output:**
```
data/dataset_txt_only_True_embeddings_True_train.zip
data/dataset_txt_only_True_embeddings_True_val.zip
```

## ① Mount Drive

In [ ]:
# Local mode - no Google Drive needed
print('Local mode')

## ② Set Working Directory

In [ ]:
import os, sys
from pathlib import Path

# Local mode - no Google Drive needed
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root")

CODE_DIR = str(PROJECT_ROOT / 'code')
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print(f'Working directory: {os.getcwd()}')

## ③ Imports

In [ ]:
import datasets
import pandas as pd
import numpy as np

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
    compute_neighbors_and_distances,
    compute_neighbor_weighted_prices_lagged,
)

## ④ Config

In [ ]:
txt_only = True
include_embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_True"

## ⑤ Load Panel Data

- `window = 28` — 28-day rolling average window
- `mod = 4` — every 4th time period (~13 non-overlapping 4-week periods)
- `max_periods = 57` — caps number of time periods

In [ ]:
window = 28
mod = 4
max_periods = 57

data_files = {
    "train": ["train-00000-of-00001.parquet"],
    "validation": ["validation-00000-of-00001.parquet"],
}

ds_dict = datasets.load_dataset(
    "parquet",
    data_dir="../data/amzn_shoes_monthly_diffs_ffill_fixed_splits",
    data_files=data_files,
)

ds_val   = ds_dict["validation"]
ds_train = ds_dict["train"]

columns = [
    "ASIN", "SALES_RANK", "PRICE", "BUYBOX_PRICE", "text", "date", "window",
    "REVIEW_COUNT", "RATING", "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
    "Lightning Deals: Upcoming Deal", "Buy Box: Is FBA", "subcat_aggregated",
]

df_val = ds_val.select_columns(columns).to_pandas()
df_val = df_val[df_val["window"] == window].dropna()
df_val["date_t"] = df_val["date"].astype("category").cat.codes
df_val = df_val[df_val["date_t"] <= max_periods]
df_val = df_val[df_val["date_t"] % mod == 0]

df_train = ds_train.select_columns(columns).to_pandas()
df_train = df_train[df_train["window"] == window].dropna()
df_train["date_t"] = df_train["date"].astype("category").cat.codes
df_train = df_train[df_train["date_t"] <= max_periods]
df_train = df_train[df_train["date_t"] % mod == 0]

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_train_val = pd.concat([df_train, df_val], axis=0).reset_index(drop=True)

for col in ['index', 'level_0']:
    if col in df_train_val.columns:
        df_train_val = df_train_val.drop(columns=[col])

num_val   = len(df_val)
num_train = len(df_train)
print(f"Number of train samples: {num_train}")
print(f"Number of val samples:   {num_val}")

df_train_val["date"] = pd.to_datetime(df_train_val["date"]).dt.strftime("%Y-%m-%d")

dummy_subcat = pd.get_dummies(df_train_val["subcat_aggregated"]) * 1
dummy_subcat_names = list(dummy_subcat.columns)
dummy_time = pd.get_dummies(df_train_val["date"]) * 1
dummy_time_names = list(dummy_time.columns)

for col in dummy_subcat.columns:
    df_train_val[col] = dummy_subcat[col].values
for col in dummy_time.columns:
    df_train_val[col] = dummy_time[col].values

df_full = df_train_val.copy()
df_full["date"] = pd.to_datetime(df_full["date"])
df_full.set_index(["ASIN", "date"], inplace=True)

assert df_full.index.is_unique, f"ERROR: index not unique!"
print(f"✅ Index is unique")
print(f"df_full shape: {df_full.shape}")

## ⑥ Load Embeddings and Predictions

In [ ]:
results_256_time_independent, configs_time_independent = load_pred_and_emb(
    embedding_size=256,
    txt_only=txt_only,
    lag_type="time_independent",
    config_path="utils/paths_config.yaml",
    get_lag2_for_diff=False,
)

results_256_lag1, configs_lag1 = load_pred_and_emb(
    embedding_size=256,
    txt_only=txt_only,
    lag_type="lag1",
    config_path="utils/paths_config.yaml",
    get_lag2_for_diff=False,
)

## ⑦ PCA, Clustering and Similarity Features

**Key fix:** `center_and_norm` concatenates train + val embeddings creating duplicates.
We deduplicate before joining to prevent row explosion in df_full.

In [ ]:
pred_and_emb = results_256_time_independent
embeddings = center_and_norm(
    pred_and_emb["embeddings"][2], pred_and_emb["embeddings"][0]
)
print(f"Embedding shape before dedup: {embeddings.shape}")

# Deduplicate — center_and_norm creates duplicate (ASIN, date) pairs
embeddings = embeddings[~embeddings.index.duplicated(keep="last")]
print(f"Embedding shape after dedup:  {embeddings.shape}")

# Align dates to datetime
embeddings.index = embeddings.index.set_levels(
    pd.to_datetime(embeddings.index.get_level_values("date").unique()), level="date"
)

_, cluster_centroids = get_cluster(embeddings, n_clusters=5, n_init=200)
pca_results = get_pca(embeddings, n_components=5)
df_pca = pd.DataFrame(
    pca_results, columns=[f"pca_{i}" for i in range(pca_results.shape[1])],
    index=embeddings.index
)
df_sim = get_similarities(embeddings, cluster_centroids)

df_full = df_full.join(df_pca)
df_full = df_full.join(df_sim)

if include_embeddings:
    df_full = df_full.join(embeddings.add_prefix("emb_"))

assert df_full.index.is_unique, "ERROR: index not unique after embeddings join!"
print(f"✅ Index still unique")
print(f"df_full shape: {df_full.shape}")
print(f"NaN in emb_0: {df_full['emb_0'].isna().sum()} / {len(df_full)}")

## ⑧ Neighbor Distances

In [ ]:
df_neighbor_asins_by_date, df_distance_df_by_date = compute_neighbors_and_distances(
    embeddings, df_full, n_neighbors=5
)

df_full = df_full.join(df_neighbor_asins_by_date)
df_full = df_full.join(df_distance_df_by_date)

print(f"df_full shape after joining neighbors: {df_full.shape}")

## ⑨ Preview

In [ ]:
df_full.head()

## ⑩ Weighted Substitute Prices (Instrument)

Distance-weighted average of 5 nearest neighbors prices lagged one period.
Key demand instrument for DoubleML in notebook 04.

In [ ]:
df_full.columns = [str(c) for c in df_full.columns]

weighted_substitute_price = compute_neighbor_weighted_prices_lagged(
    df_full, "BUYBOX_PRICE"
)
df_full = df_full.join(weighted_substitute_price)
df_full.loc[:, ["BUYBOX_PRICE", "weighted_substitute_price_lagged"]].head()

## ⑪ Split Back into Train and Val

In [ ]:
df_full_val   = df_full.iloc[num_train:, :].copy()
df_full_train = df_full.iloc[:num_train, :].copy()

print(f"df_full shape:       {df_full.shape}")
print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape:   {df_full_val.shape}")

## ⑫ NaN Check

In [ ]:
print(df_full_val.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
print(df_full_train.isna().sum().sort_values(ascending=False).head(20))

## ⑬ Add Lag Features and Join Model Predictions

**Key fix:** Convert prediction dates to datetime before joining.
This ensures the ASIN+date index matches df_full_train and df_full_val.

In [ ]:
df_full_val = df_full_val.rename(
    columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"}
)
for i in range(1, 3):
    df_full_val[f"Q_t-{i}"]            = df_full_val.groupby("ASIN")["Q_t"].shift(i)
    df_full_val[f"P_bb_t-{i}"]         = df_full_val.groupby("ASIN")["P_bb_t"].shift(i)
    df_full_val[f"REVIEW_COUNT_t-{i}"] = df_full_val["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_val[f"RATING_t-{i}"]       = df_full_val["RATING"].groupby("ASIN").shift(i)

df_full_val["Delta_Q_t"]    = df_full_val["Q_t"]    - df_full_val["Q_t-1"]
df_full_val["Delta_P_bb_t"] = df_full_val["P_bb_t"] - df_full_val["P_bb_t-1"]

df_full_train = df_full_train.rename(
    columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"}
)
for i in range(1, 3):
    df_full_train[f"Q_t-{i}"]            = df_full_train["Q_t"].groupby("ASIN").shift(i)
    df_full_train[f"P_bb_t-{i}"]         = df_full_train["P_bb_t"].groupby("ASIN").shift(i)
    df_full_train[f"REVIEW_COUNT_t-{i}"] = df_full_train["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_train[f"RATING_t-{i}"]       = df_full_train["RATING"].groupby("ASIN").shift(i)

df_full_train["Delta_Q_t"]    = df_full_train["Q_t"]    - df_full_train["Q_t-1"]
df_full_train["Delta_P_bb_t"] = df_full_train["P_bb_t"] - df_full_train["P_bb_t-1"]

# KEY FIX: convert prediction dates to datetime before joining
# This ensures the ASIN+date index matches df_full_train/val datetime index
def prep_pred(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index(["ASIN", "date"])

val_predictions_levl   = prep_pred(pred_and_emb["predictions"][0])
val_predictions_diff   = prep_pred(pred_and_emb["predictions"][1]).rename(
    columns={"pred_ml_l": "pred_ml_l_diff", "pred_ml_m": "pred_ml_m_diff"}
)
train_predictions_levl = prep_pred(pred_and_emb["predictions"][2])
train_predictions_diff = prep_pred(pred_and_emb["predictions"][3]).rename(
    columns={"pred_ml_l": "pred_ml_l_diff", "pred_ml_m": "pred_ml_m_diff"}
)

df_full_val   = df_full_val.join(val_predictions_levl)
df_full_train = df_full_train.join(train_predictions_levl)
df_full_val   = df_full_val.join(val_predictions_diff)
df_full_train = df_full_train.join(train_predictions_diff)

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape:   {df_full_val.shape}")
print(f"NaN in pred_ml_l train: {df_full_train['pred_ml_l'].isna().sum()} / {len(df_full_train)}")
print(f"NaN in pred_ml_l val:   {df_full_val['pred_ml_l'].isna().sum()} / {len(df_full_val)}")

## ⑭ Join Lag1 Predictions

In [ ]:
val_predictions_levl_lag1   = prep_pred(results_256_lag1["predictions"][0]).rename(
    columns={"pred_ml_l": "pred_ml_l_lag_1", "pred_ml_m": "pred_ml_m_lag_1"}
)
val_predictions_diff_lag1   = prep_pred(results_256_lag1["predictions"][1]).rename(
    columns={"pred_ml_l": "pred_ml_l_diff_lag_1", "pred_ml_m": "pred_ml_m_diff_lag_1"}
)
train_predictions_levl_lag1 = prep_pred(results_256_lag1["predictions"][2]).rename(
    columns={"pred_ml_l": "pred_ml_l_lag_1", "pred_ml_m": "pred_ml_m_lag_1"}
)
train_predictions_diff_lag1 = prep_pred(results_256_lag1["predictions"][3]).rename(
    columns={"pred_ml_l": "pred_ml_l_diff_lag_1", "pred_ml_m": "pred_ml_m_diff_lag_1"}
)

df_full_val   = df_full_val.join(val_predictions_levl_lag1)
df_full_train = df_full_train.join(train_predictions_levl_lag1)
df_full_val   = df_full_val.join(val_predictions_diff_lag1)
df_full_train = df_full_train.join(train_predictions_diff_lag1)

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape:   {df_full_val.shape}")

## ⑮ Check Unique Dates

In [ ]:
df_full_val.reset_index()["date"].dropna().unique()

## ⑯ Drop Redundant Columns

In [ ]:
drop_cols = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
    "pred_ml_l_lag_1", "pred_ml_m_lag_1",
]
keep_cols = [c for c in df_full_val.columns if c not in drop_cols]

## ⑰ Final NaN Check

In [ ]:
print(df_full_val[keep_cols].isna().sum().sort_values(ascending=False).head(20))
print("\n\n")
print(df_full_train[keep_cols].isna().sum().sort_values(ascending=False).head(20))

## ⑱ Rename Weighted Substitute Price Column

In [ ]:
df_full_val = df_full_val.rename(
    columns={"weighted_substitute_price_lagged": "weighted_substitute_price"}
)
df_full_train = df_full_train.rename(
    columns={"weighted_substitute_price_lagged": "weighted_substitute_price"}
)

## ⑲ Inspect Final Columns

In [ ]:
list(df_full_val.columns)

In [ ]:
list(df_full_train.columns)

## ⑳ Save Output

**Output files (input to notebooks 03_2 and 04):**
```
data/dataset_txt_only_True_embeddings_True_train.zip
data/dataset_txt_only_True_embeddings_True_val.zip
```

In [ ]:
zip_train = dict(method="zip", archive_name=f"../data/{dataframe_name}_train.csv")
df_full_train.to_csv(f"../data/{dataframe_name}_train.zip", compression=zip_train)
zip_val = dict(method="zip", archive_name=f"../data/{dataframe_name}_val.csv")
df_full_val.to_csv(f"../data/{dataframe_name}_val.zip", compression=zip_val)
print(f"✅ Saved: {dataframe_name}_train.zip")
print(f"✅ Saved: {dataframe_name}_val.zip")